In [ ]:
# Import necessary libraries
import pandas as pd
import os
import glob
from pathlib import Path

# Directory containing the parquet files - adjust to your local path
DATA_DIR = 'INPUT DIRECTORY HERE'
# Directory to save the filtered data
OUTPUT_DIR = 'OUTPUT DIRECTORY HERE'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the column names based on the information provided
MARKET_CAP_COL = 'mktcap_lag'
FIRM_ID_COL = 'permno'

def get_bottom_firms(percentage=0.2):
    """
    Identify the bottom percentage of firms by market cap in 2016
    """
    print(f"Loading data from year 2016...")
    
    # Load the 2016 data
    df_2016 = pd.read_parquet(f'{DATA_DIR}/year_2016.parquet')
    
    # Check if we have the required columns
    if MARKET_CAP_COL not in df_2016.columns:
        raise ValueError(f"Market cap column '{MARKET_CAP_COL}' not found in the data")
    if FIRM_ID_COL not in df_2016.columns:
        raise ValueError(f"Firm ID column '{FIRM_ID_COL}' not found in the data")
    
    # Print info about the dataset
    print(f"Loaded 2016 data with {len(df_2016)} rows")
    print(f"Using '{MARKET_CAP_COL}' as the market cap column")
    print(f"Using '{FIRM_ID_COL}' as the firm identifier column")
    
    # Group by firm to get the most recent market cap for each firm
    # If the data is monthly, we want to use the most recent month for each firm
    if 'month' in df_2016.columns:
        # Sort by month to get the most recent month's data for each firm
        df_2016 = df_2016.sort_values('month', ascending=False)
        # Get unique firms with their most recent market cap
        df_firms = df_2016.drop_duplicates(FIRM_ID_COL, keep='first')
        print(f"Found {len(df_firms)} unique firms after grouping by most recent month")
    else:
        # If no month column, assume one entry per firm
        df_firms = df_2016
        print(f"No month column found, assuming one entry per firm")
    
    # Print market cap statistics
    print(f"\nMarket cap column statistics:")
    print(df_firms[MARKET_CAP_COL].describe())
    
    # Identify firms in the bottom percentage by market cap
    # First, drop rows with missing market cap
    df_firms_valid = df_firms.dropna(subset=[MARKET_CAP_COL])
    print(f"Firms with valid market cap: {len(df_firms_valid)}")
    
    # Calculate the threshold for bottom percentage
    threshold = df_firms_valid[MARKET_CAP_COL].quantile(percentage)
    print(f"\nMarket cap threshold (bottom {percentage*100}%): {threshold}")
    
    # Get the firms below the threshold
    bottom_firms = df_firms_valid[df_firms_valid[MARKET_CAP_COL] <= threshold][FIRM_ID_COL].unique()
    print(f"Number of firms in bottom {percentage*100}%: {len(bottom_firms)}")
    print(f"Total number of unique firms: {df_firms_valid[FIRM_ID_COL].nunique()}")
    
    # Return as a set for faster lookups
    return set(bottom_firms)

def filter_all_years(bottom_firms):
    """
    Filter out the bottom firms from all years
    """
    # Get all parquet files
    parquet_files = sorted(glob.glob(f'{DATA_DIR}/year_*.parquet'))
    
    print(f"\nFound {len(parquet_files)} parquet files")
    
    # Process each file
    for file_path in parquet_files:
        year_file = Path(file_path).name
        print(f"Processing {year_file}...")
        
        # Load the data
        df = pd.read_parquet(file_path)
        
        # Original row count and firm count
        original_count = len(df)
        original_firms = df[FIRM_ID_COL].nunique()
        
        # Filter out the bottom firms
        df_filtered = df[~df[FIRM_ID_COL].isin(bottom_firms)]
        
        # New row count and firm count
        new_count = len(df_filtered)
        new_firms = df_filtered[FIRM_ID_COL].nunique()
        
        rows_removed = original_count - new_count
        firms_removed = original_firms - new_firms
        
        print(f"  Removed {rows_removed} rows ({rows_removed/original_count:.2%}) from {year_file}")
        print(f"  Removed {firms_removed} firms out of {original_firms} ({firms_removed/original_firms:.2%})")
        
        # Save the filtered data
        output_path = f'{OUTPUT_DIR}/{year_file}'
        df_filtered.to_parquet(output_path)
        print(f"  Saved filtered data to {output_path}")

if __name__ == "__main__":
    # Execute the filtering
    bottom_firms = get_bottom_firms(0.2)  # Get bottom 20%

    # Confirm before proceeding
    proceed = input(f"\nProceed with filtering {len(bottom_firms)} firms from all years? (y/n): ")
    if proceed.lower() == 'y':
        filter_all_years(bottom_firms)
        print("\nProcessing complete. Filtered data saved to:", OUTPUT_DIR)
    else:
        print("Operation cancelled.")

Loading data from year 2016...
Loaded 2016 data with 43765 rows
Using 'mktcap_lag' as the market cap column
Using 'permno' as the firm identifier column
Found 3868 unique firms after grouping by most recent month

Market cap column statistics:
count      3868.000000
mean       6042.126690
std       25297.425821
min           0.913200
25%         123.165797
50%         622.089190
75%        2750.850847
max      589327.232760
Name: mktcap_lag, dtype: float64
Firms with valid market cap: 3868

Market cap threshold (bottom 20.0%): 85.4409
Number of firms in bottom 20.0%: 774
Total number of unique firms: 3868

Found 60 parquet files
Processing year_1957.parquet...
  Removed 11 rows (0.20%) from year_1957.parquet
  Removed 1 firms out of 494 (0.20%)
  Saved filtered data to /Users/marcusdehaan/Desktop/oracle/df_filtered/year_1957.parquet
Processing year_1958.parquet...
  Removed 12 rows (0.20%) from year_1958.parquet
  Removed 1 firms out of 502 (0.20%)
  Saved filtered data to /Users/marcu

In [1]:
import pandas as pd

# Replace the path with the correct local path if needed
df = pd.read_parquet('./df_filtered/year_1999.parquet')

print(df.columns.tolist())


['permno', 'month', 'mktcap_lag', 'ret_excess', 'sic2_1.0', 'sic2_2.0', 'sic2_7.0', 'sic2_8.0', 'sic2_10.0', 'sic2_12.0', 'sic2_13.0', 'sic2_14.0', 'sic2_15.0', 'sic2_16.0', 'sic2_17.0', 'sic2_20.0', 'sic2_21.0', 'sic2_22.0', 'sic2_23.0', 'sic2_24.0', 'sic2_25.0', 'sic2_26.0', 'sic2_27.0', 'sic2_28.0', 'sic2_29.0', 'sic2_30.0', 'sic2_31.0', 'sic2_32.0', 'sic2_33.0', 'sic2_34.0', 'sic2_35.0', 'sic2_36.0', 'sic2_37.0', 'sic2_38.0', 'sic2_39.0', 'sic2_40.0', 'sic2_41.0', 'sic2_42.0', 'sic2_44.0', 'sic2_45.0', 'sic2_46.0', 'sic2_47.0', 'sic2_48.0', 'sic2_49.0', 'sic2_50.0', 'sic2_51.0', 'sic2_52.0', 'sic2_53.0', 'sic2_54.0', 'sic2_55.0', 'sic2_56.0', 'sic2_57.0', 'sic2_58.0', 'sic2_59.0', 'sic2_60.0', 'sic2_61.0', 'sic2_62.0', 'sic2_63.0', 'sic2_64.0', 'sic2_65.0', 'sic2_67.0', 'sic2_70.0', 'sic2_72.0', 'sic2_73.0', 'sic2_75.0', 'sic2_76.0', 'sic2_78.0', 'sic2_79.0', 'sic2_80.0', 'sic2_81.0', 'sic2_82.0', 'sic2_83.0', 'sic2_84.0', 'sic2_86.0', 'sic2_87.0', 'sic2_89.0', 'sic2_99.0', 'charac